# Phase 2 — explicit-solvent MD on a GPU

Run this on **Google Colab with a T4 GPU**: `Runtime` → `Change runtime type` →
`T4 GPU`.

## Why this notebook exists

Phase 1 ran short implicit-solvent MD on a laptop and produced a clear negative
result:

| Model | reduced chi-squared | Rg |
|---|---|---|
| experiment (SASBDB `SASDMJ2`) | — | 13.98 A |
| crystal structure `2vb1` | **1.67** | 14.00 A |
| implicit-solvent MD | **14.5 – 20.2** | 14.2 – 14.4 A |

The simulation expanded lysozyme away from a crystal structure that already
agreed with experiment. The expansion appeared within 2 ps and was identical at
nonbonded cutoffs of 2.0, 1.5, 1.2 and 1.0 nm, so it is a systematic bias of
the GB solvent model rather than a settings artifact.

That leaves one question Phase 1 could not answer: **is the disagreement the
solvent model, or the force field?** Only explicit solvent separates them, and
explicit solvent needs a GPU — solvating this protein takes it from 1,960 to
about 22,500 atoms.

## What this does not settle

If explicit solvent recovers the experimental Rg, the GB model was at fault. If
it does not, the problem lies deeper — in the force field, the starting
structure, or the assumption that one short trajectory represents the
solution ensemble. Either outcome is informative; neither is guaranteed.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Install

OpenMM's PyPI wheels carry CUDA support, so `pip` is enough here — no conda
needed. (On Windows those same wheels conflict with Anaconda's runtime, which
is why the local environment uses conda-forge instead. Linux has no such
problem.)

In [ ]:
!pip install -q openmm pdbfixer mdtraj biopython requests
!git clone -q https://github.com/anandswadurkar/Computational-Small-Angle-X-ray-Scattering-SAXS-and-Nuclear-Magnetic-Resonance-NMR-for-Biomolecules.git saxsnmr_repo || echo "already cloned"
import sys
sys.path.insert(0, "/content/saxsnmr_repo")

import openmm as mm
print("openmm", mm.__version__)
print("platforms:", [mm.Platform.getPlatform(i).getName()
                     for i in range(mm.Platform.getNumPlatforms())])

In [ ]:
# Confirm CUDA is actually usable, not merely listed.
import openmm as mm
s = mm.System(); s.addParticle(1.0)
ctx = mm.Context(s, mm.VerletIntegrator(0.001), mm.Platform.getPlatformByName("CUDA"))
print("CUDA context OK:", ctx.getPlatform().getName())
del ctx

## 3. Google Drive (optional but recommended)

Colab disconnects — on idle, on timeout, and sometimes for no stated reason. A
checkpoint on Drive lets the run resume instead of restarting. Skip this cell
if you are happy to lose an interrupted run.

In [ ]:
USE_DRIVE = True   # set False to keep everything on the ephemeral local disk

from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/saxsnmr_phase2")
else:
    WORK = Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
print("working directory:", WORK)

## 4. Fetch and prepare

Identical to Phase 1 — the same fetch-and-join layer, the same structure
preparation. Only the solvent and the hardware change.

In [ ]:
from saxsnmr import Fetcher, prepare

UNIPROT = "P00698"          # hen egg white lysozyme
CODE = "SASDMJ2"            # monomeric, q to 0.495 1/A

fetcher = Fetcher(cache_dir=WORK / "cache")
curve = fetcher.sasbdb_curve(CODE).trim(q_max=0.50)
print(curve)

structure, provenance = fetcher.structure_for(UNIPROT)
print("structure:", provenance)

prep = prepare(structure, WORK / "prepared.pdb", chain="A")
print(f"{prep.n_residues} residues, {prep.n_atoms} atoms")

## 5. Solvate

A 1 nm padding keeps the protein from seeing its own periodic image under a
1 nm cutoff without paying for water that does nothing. Expect roughly 22,500
atoms and a 6.2 nm box.

This step is CPU-bound and single-threaded. It took 48 minutes on the laptop
this project was developed on; Colab is considerably faster, but it is not
instant. The result is cached, so a reconnect does not repeat it.

In [ ]:
from saxsnmr import solvate

# reuse=True (the default) returns an existing box instead of rebuilding it,
# so a reconnect does not repeat the slow part.
solv = solvate(prep, WORK / "solvated.pdb", padding_nm=1.0, ionic_strength_M=0.15)
print(solv)
print(f"growth: {solv.n_atoms / prep.n_atoms:.1f}x")

## 6. Explicit-solvent MD

PME electrostatics, NPT with a Monte Carlo barostat, 4 fs timestep with
hydrogen mass repartitioning.

Only protein coordinates are written. Water is most of the system but none of
it reaches the SAXS calculator, which models the hydration shell through its
fitted contrast parameter — storing it would multiply the file size roughly
tenfold for no benefit.

Start with 2 ns. On a T4 that is well inside a Colab session; raise it once you
see the reported speed.

`production_ns` is a **cumulative target, not an amount to add**. Re-running
this cell after a disconnect resumes from the checkpoint and keeps the frames
already recorded; raising `production_ns` and re-running extends the
trajectory to the new length. Asking for a length already reached does
nothing. To start over instead, delete the `md` directory.

In [ ]:
from saxsnmr import run_md_explicit

result = run_md_explicit(
    solv, WORK / "md",
    production_ns=2.0,
    equilibration_ps=200.0,
    frame_interval_ps=10.0,
    platform="CUDA",
    checkpoint=WORK / "md" / "state.chk",
)
print()
print(result)

## 7. Did explicit solvent recover the experimental Rg?

This is the question the whole notebook exists to answer. Phase 1's
implicit-solvent run sat at 14.2–14.4 A against an experimental 13.98 A.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from saxsnmr import load_trajectory, radius_of_gyration

traj = load_trajectory(result.trajectory, top=result.topology)
rg = radius_of_gyration(traj)
# Derived, not hard-coded, so the axis stays right if you change the interval
# or extend the run.
frame_ps = result.simulated_ps / max(result.n_frames, 1)
t_ps = np.arange(len(rg)) * frame_ps

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t_ps, rg, lw=1, label=f"explicit solvent: {rg.mean():.2f} +/- {rg.std():.2f} A")
ax.axhline(curve.guinier_rg * 10, color="crimson", ls="--",
           label=f"experiment (Guinier): {curve.guinier_rg * 10:.2f} A")
ax.axhspan(14.2, 14.4, color="grey", alpha=0.25, label="Phase 1 implicit solvent")
ax.set_xlabel("time (ps)")
ax.set_ylabel("Rg (A)")
ax.legend(fontsize=8)
plt.tight_layout()

print(f"explicit  Rg = {rg.mean():.2f} +/- {rg.std():.2f} A")
print(f"implicit  Rg = 14.2 - 14.4 A   (Phase 1)")
print(f"experiment   = {curve.guinier_rg * 10:.2f} A")

## 8. The SAXS fit

In [ ]:
from saxsnmr import ensemble_fit, frame_tables
from saxsnmr.saxs import fit

static = load_trajectory(prep.path)
crystal = fit(frame_tables(static, 0), curve.q, curve.intensity, curve.sigma)
ens = ensemble_fit(result, curve, stride=1)

print(f"crystal structure     : chi2 = {crystal.chi2:7.3f}  c1={crystal.c1:.3f} c2={crystal.c2:.3f}")
print(f"explicit-solvent MD   : chi2 = {ens.chi2:7.3f}  c1={ens.c1:.3f} c2={ens.c2:.3f}")
print(f"implicit-solvent MD   : chi2 =  14.5 - 20.2        (Phase 1)")

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(curve.q, curve.intensity, yerr=ens.sigma, fmt=".", ms=3, lw=0.5,
            alpha=0.5, label="experiment")
ax.plot(curve.q, crystal.model, lw=1.2, label=f"crystal, chi2={crystal.chi2:.2f}")
ax.plot(curve.q, ens.model, lw=1.5, label=f"explicit MD, chi2={ens.chi2:.2f}")
ax.set_yscale("log"); ax.set_ylabel("I(q)"); ax.legend()

axr.axhline(0, color="k", lw=0.5)
axr.plot(curve.q, (ens.model - curve.intensity) / ens.sigma, lw=1)
axr.set_xlabel("q  (1/A)"); axr.set_ylabel("residual / sigma")
plt.tight_layout()

## 9. Save the trajectory

Keep the protein-only DCD and its topology. Together they are a few megabytes
and everything the local analysis needs — the full solvated trajectory would be
an order of magnitude larger and is not used.

In [ ]:
import shutil
out = WORK / "phase2_results"
out.mkdir(exist_ok=True)
shutil.copy(result.trajectory, out / "production_protein.dcd")
shutil.copy(result.topology, out / "topology_protein.pdb")

sizes = {p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in out.iterdir()}
print("saved to", out)
for k, v in sizes.items():
    print(" ", k, v)

# Uncomment to download rather than relying on Drive:
# from google.colab import files
# files.download(str(out / "production_protein.dcd"))

## 10. Reading the result

**If Rg came back near 13.98 A and chi-squared dropped towards the crystal
structure's 1.67**, the GB implicit solvent was the problem, and Phase 1's
negative result was a limitation of the approximation rather than of the force
field.

**If Rg stayed near 14.2–14.4 A**, explicit solvent did not rescue it, and the
disagreement lies elsewhere: the force field, the starting structure, or the
premise that a single short trajectory represents a solution ensemble. Worth
checking before concluding anything — 2 ns is short, and the equilibration may
simply not be finished.

**If chi-squared improved but stayed well above the crystal structure**, the
honest reading is that MD is not adding information here. A single
high-resolution crystal structure already describes this protein's solution
scattering well, which is a perfectly reasonable thing to be true of a small,
rigid, well-ordered protein. Flexible or multidomain systems are where an
ensemble earns its cost.

Either way, report the fitted `c1` and `c2` alongside chi-squared. Two free
parameters absorb a great deal.